# 03 — Delivery feature engineering
Create features that are plausibly known when an order is placed. We exclude pickup time and multiple deliveries from this first model because they may be unavailable at checkout or reflect later operations.

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
PROCESSED = ROOT/"data/processed"
delivery = pd.read_csv(PROCESSED/"delivery_train_clean.csv", low_memory=False)

def parse_order_time(value):
    if pd.isna(value): return np.nan, np.nan
    m = re.match(r"^\s*(\d{1,2}):(\d{2})(?::\d{2})?\s*(AM|PM)?\s*$", str(value), re.I)
    if not m: return np.nan, np.nan
    hour, minute = int(m.group(1)), int(m.group(2))
    suffix = (m.group(3) or "").upper()
    if suffix == "PM" and hour < 12: hour += 12
    if suffix == "AM" and hour == 12: hour = 0
    if hour > 23 or minute > 59: return np.nan, np.nan
    return hour, minute

def build_features(df):
    x = df.copy()
    numeric = ["Delivery_person_Age", "Delivery_person_Ratings", "Restaurant_latitude",
               "Restaurant_longitude", "Delivery_location_latitude",
               "Delivery_location_longitude", "Vehicle_condition"]
    for c in numeric:
        x[c] = pd.to_numeric(x[c], errors="coerce")

    # Haversine straight-line distance, not road distance.
    lat1, lon1 = np.radians(x["Restaurant_latitude"]), np.radians(x["Restaurant_longitude"])
    lat2, lon2 = np.radians(x["Delivery_location_latitude"]), np.radians(x["Delivery_location_longitude"])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    x["distance_km"] = 6371 * 2*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    hm = x["Time_Orderd"].apply(parse_order_time)
    x["order_hour"] = hm.map(lambda v: v[0])
    x["order_minute"] = hm.map(lambda v: v[1])
    dt = pd.to_datetime(x["Order_Date"], errors="coerce", dayfirst=True)
    x["order_weekday"] = dt.dt.dayofweek
    x["order_month"] = dt.dt.month

    numeric_features = numeric + ["distance_km", "order_hour", "order_minute", "order_weekday", "order_month"]
    categorical_features = ["Weatherconditions", "Road_traffic_density", "Type_of_order",
                            "Type_of_vehicle", "Festival", "City"]
    for c in categorical_features:
        x[c] = x[c].astype("string").str.strip().replace(
            {"": pd.NA, "NaN": pd.NA, "nan": pd.NA, "conditions NaN": pd.NA}
        ).astype(object)
        x[c] = x[c].where(pd.notna(x[c]), np.nan)
    features = numeric_features + categorical_features
    return x[features], numeric_features, categorical_features

X, numeric_features, categorical_features = build_features(delivery)
y = pd.to_numeric(delivery["target_delivery_minutes"], errors="coerce")
valid = y.notna() & y.between(1, 180)
X, y = X.loc[valid], y.loc[valid]
print("X:", X.shape, "| y:", y.shape)
display(X.head())
display(X.isna().sum().sort_values(ascending=False).to_frame("missing"))

X: (45593, 18) | y: (45593,)


/var/folders/44/fywnm5sj335fdzvrq6s4_m3c0000gn/T/ipykernel_10567/4002191272.py:40: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  x["order_weekday"] = dt.dt.dayofweek


,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Vehicle_condition,distance_km,order_hour,order_minute,order_weekday,order_month,Weatherconditions,Road_traffic_density,Type_of_order,Type_of_vehicle,Festival,City
0,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2,3.025149,11.0,30.0,5,3,conditions Sunny,High,Snack,motorcycle,No,Urban
1,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2,20.183530,19.0,45.0,4,3,conditions Stormy,Jam,Snack,scooter,No,Metropolitian
2,23.0,4.4,12.914264,77.678400,12.924264,77.688400,0,1.552758,8.0,30.0,5,3,conditions Sandstorms,Low,Drinks,motorcycle,No,Urban
3,38.0,4.7,11.003669,76.976494,11.053669,77.026494,0,7.790401,18.0,0.0,1,4,conditions Sunny,Medium,Buffet,motorcycle,No,Metropolitian
4,32.0,4.6,12.972793,80.249982,13.012793,80.289982,1,6.210138,13.0,30.0,5,3,conditions Cloudy,High,Snack,scooter,No,Metropolitian


,missing
distance_km,3640
Restaurant_latitude,3640
Restaurant_longitude,3640
Delivery_person_Ratings,1908
Delivery_person_Age,1854
order_hour,1731
order_minute,1731
City,1200
Weatherconditions,616
Road_traffic_density,601


In [2]:
table = X.copy()
table["target_delivery_minutes"] = y
table.to_csv(PROCESSED/"delivery_features.csv", index=False)
print("Saved", PROCESSED/"delivery_features.csv")

Saved /Users/kakulrajput/Documents/food-ordering-delivery-ml/data/processed/delivery_features.csv
